# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates step-by-step loading, exploration, and analysis of the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the `mlcroissant` library, following the Croissant metadata standard.

### Dataset Source
Croissant schema source: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`. The metadata object provides a programmatic interface to the dataset documentation and schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant-compliant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview

Explore the available record sets and their fields via their `@id`.

- List all record sets, their `@id`, and field `@id`s
- This overview enables you to reference and extract data precisely.

In [ ]:
# List all record sets in the dataset, identified by `@id`
record_sets = metadata.record_sets
if not record_sets:
    print("This metadata schema does not contain any record sets.")
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"- Record Set Name: {rs.name}\n  @id: {rs.id}")
      
        # List all fields (by @id) for this record set
        if hasattr(rs, "fields") and rs.fields:
            print("  Fields:")
            for field in rs.fields:
                print(f"    - {field.id} (name: {getattr(field, 'name', '<no name>')})")
        print()

## 3. Data Extraction

Load data from each record set into pandas DataFrames using their `@id` as reference. This enables further analysis using standard tools.

`mlcroissant.Dataset.records(record_set=<record_set_id>)` returns a generator over the record set rows.

In [ ]:
# 1. Get all record set @id
record_sets = metadata.record_sets
record_set_ids = [rs.id for rs in record_sets]

# 2. Load all record sets as DataFrames using their @id
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set: {rs_id}, {df.shape[0]} rows, {df.shape[1]} columns.")
    else:
        print(f"Record set {rs_id} contains no rows.")

# Show one example DataFrame (if any were loaded)
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Available columns in record set '{first_rs_id}':\n{dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Below is an example of EDA for a numeric variable in a chosen record set. All fields are referenced by their `@id`. Replace `<REPLACE_ME>` placeholders to match field `@id` values as listed above.

In [ ]:
# ----- EDA: Numeric Field Processing -----

# Pick one record set (use the first loaded above)
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Display field (column) @id list
    print(f"Available columns (@id) in '{record_set_id}':\n{df.columns.tolist()}")
    
    # Choose a likely numeric field by @id (e.g., 'schema:age', replace with actual if different)
    # If unsure, default to the first float/int column found
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        print("No numeric field found for EDA.")
    else:
        threshold = df[numeric_field].quantile(0.1)  # 10th percentile as a sample threshold

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization (z-score)
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Try grouping by another field (by @id): pick first non-numeric column
        group_field = None
        for col in df.columns:
            if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable categorical/grouping field found.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize the distribution of the normalized numeric field, and the mean values grouped by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    # Histogram of the normalized numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[normalized_col], bins=15, kde=True, color='coral')
    plt.title(f"Distribution of Normalized {numeric_field} in '{record_set_id}'")
    plt.xlabel(f"{normalized_col}")
    plt.ylabel("Count")
    plt.show()

    # Barplot of grouped means
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field, palette='Blues_r')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion

- We have loaded the FAIR² dataset metadata and records using `mlcroissant` and reviewed its record sets using the Croissant `@id` fields for precision.
- Data was extracted, filtered, normalized, and grouped by the designated schema `@id`s, allowing reproducible analysis across versioned FAIR datasets.
- Visualizations help reveal both the distribution and groupwise dynamics of clinical variables, supporting downstream research questions on second primary colorectal cancer in survivors.

For further analysis, consult the record set and field `@id`s to select and explore variables most relevant to your research task.